<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m3_memory_manager.ipynb)
[![Course](https://img.shields.io/badge/Full%20Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-3)

</div>

# Module 3 — State, Memory & Context Engineering

**Level:** Medium-Advanced | **Time:** ~1.5h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-3)

### What you'll build
A `MemoryManager` with three memory tiers: in-context rolling summary, episodic vector retrieval, and semantic fact triples.

### Key concepts
- **Context window cost model**: `Σ input_tokens × price_per_token` per step — context is the main cost lever
- **3 memory tiers**: in-context (short-term) → episodic (long-term, vector-indexed) → semantic (distilled facts)
- **Context rot**: quality degrades after ~20 turns if you naively append all history
- **Session handoff**: initializer writes digest; executor reads it on startup
- **Two-agent memory separation** (Anthropic pattern): Initializer writes, Executor reads + extends

### Research refs
- Generative Agents — Park et al. (2023) https://arxiv.org/abs/2304.03442
- MemGPT — Packer et al. (2023) https://arxiv.org/abs/2310.08560
- Voyager — Wang et al. (2023) https://arxiv.org/abs/2305.16291

---

In [ ]:
!pip install openai --quiet

In [ ]:
import time
import json
from dataclasses import dataclass, field
from typing import Optional
import math

# ── In-memory vector similarity (no external deps) ────────────────────────────

def cosine_sim(a: list[float], b: list[float]) -> float:
    dot = sum(x*y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x*x for x in a))
    mag_b = math.sqrt(sum(x*x for x in b))
    return dot / (mag_a * mag_b + 1e-10)

def mock_embed(text: str) -> list[float]:
    """Deterministic mock embedding based on char codes. Replace with real embeddings."""
    import hashlib
    h = hashlib.sha256(text.encode()).digest()
    return [((b / 255.0) - 0.5) * 2 for b in h[:32]]


# ── Memory tiers ──────────────────────────────────────────────────────────────

@dataclass
class EpisodicMemory:
    user_id: str
    timestamp: float
    text: str
    embedding: list[float]

@dataclass
class SemanticFact:
    subject: str
    predicate: str
    object_: str
    confidence: float = 1.0

class MemoryManager:
    """
    Three-tier memory system:
    - Tier 1 (in-context): rolling summary of recent turns
    - Tier 2 (episodic): vector-indexed past sessions, retrieved by relevance
    - Tier 3 (semantic): distilled fact triples, append-only
    """

    MAX_IN_CONTEXT_TURNS = 8
    MAX_EPISODIC_RESULTS = 3

    def __init__(self, user_id: str):
        self.user_id = user_id
        self._turns: list[dict] = []           # Tier 1: recent turns
        self._episodic: list[EpisodicMemory] = []  # Tier 2: past sessions
        self._semantic: list[SemanticFact] = []    # Tier 3: fact triples
        self._summary: str = ""                # Condensed history

    # ── Tier 1: In-context ────────────────────────────────────────────────────

    def add_turn(self, role: str, content: str):
        self._turns.append({"role": role, "content": content})
        if len(self._turns) > self.MAX_IN_CONTEXT_TURNS:
            self._compress_oldest()

    def _compress_oldest(self):
        """Summarize oldest 4 turns into a running summary."""
        oldest = self._turns[:4]
        summary_fragment = "; ".join(
            f"{t['role']}: {t['content'][:60]}..." for t in oldest
        )
        self._summary = f"{self._summary} | {summary_fragment}".strip(" |")
        self._turns = self._turns[4:]

    # ── Tier 2: Episodic ──────────────────────────────────────────────────────

    def store_episode(self, text: str):
        emb = mock_embed(text)
        self._episodic.append(EpisodicMemory(
            user_id=self.user_id,
            timestamp=time.time(),
            text=text,
            embedding=emb
        ))

    def retrieve_episodes(self, query: str, max_tokens: int = 500) -> list[str]:
        if not self._episodic:
            return []
        q_emb = mock_embed(query)
        scored = sorted(
            self._episodic,
            key=lambda e: cosine_sim(q_emb, e.embedding),
            reverse=True
        )
        results, total = [], 0
        for ep in scored[:self.MAX_EPISODIC_RESULTS]:
            tokens_est = len(ep.text.split()) * 1.3
            if total + tokens_est > max_tokens:
                break
            results.append(ep.text)
            total += tokens_est
        return results

    # ── Tier 3: Semantic ──────────────────────────────────────────────────────

    def add_fact(self, subject: str, predicate: str, object_: str, confidence: float = 1.0):
        self._semantic.append(SemanticFact(subject, predicate, object_, confidence))

    def get_facts(self, subject: str) -> list[SemanticFact]:
        return [f for f in self._semantic if f.subject.lower() == subject.lower()]

    # ── Context assembly ──────────────────────────────────────────────────────

    def get_context(self, query: str, max_tokens: int = 1000) -> str:
        parts = []
        if self._summary:
            parts.append(f"[Session history] {self._summary}")
        episodes = self.retrieve_episodes(query, max_tokens=400)
        if episodes:
            parts.append("[Past sessions]\n" + "\n".join(f"- {e}" for e in episodes))
        facts = self._semantic
        if facts:
            parts.append("[Known facts]\n" + "\n".join(
                f"- {f.subject} {f.predicate} {f.object_}" for f in facts[:10]
            ))
        return "\n\n".join(parts) if parts else ""


# ── Demo ──────────────────────────────────────────────────────────────────────

mem = MemoryManager(user_id="user_123")

# Simulate a session
for i in range(6):
    mem.add_turn("user", f"Tell me about transformer architecture part {i}")
    mem.add_turn("assistant", f"Transformers use self-attention. Part {i} key insight: ...")

# Store past session episodes
mem.store_episode("User asked about BERT fine-tuning on sentiment analysis. Used AdamW, lr=2e-5.")
mem.store_episode("User built a RAG pipeline with Pinecone. Chunk size 512, overlap 64.")
mem.store_episode("User asked about GPT-2 tokenizer. BPE, 50k vocab, cl100k_base.")

# Add semantic facts
mem.add_fact("user", "prefers", "PyTorch over JAX")
mem.add_fact("user", "works_at", "startup building LLM-powered search")

# Retrieve context for a new query
ctx = mem.get_context("How do I fine-tune a language model?")
print("Retrieved context:")
print(ctx)
print(f"\nIn-context turns: {len(mem._turns)}")
print(f"Episodic memories: {len(mem._episodic)}")
print(f"Semantic facts: {len(mem._semantic)}")

## Exercise

Design the memory retrieval strategy for a multi-session coding agent.

> **Interview question:** *"An agent is degrading after 20 turns in production. How do you diagnose and fix context rot?"*

In [ ]:
# Exercise: Memory strategy for a multi-session coding agent
#
# Design retrieve_context() for an agent that helps a user build a Python project
# across 10+ sessions over 2 weeks.
#
# Questions to answer:
# 1. What goes in Tier 1 (verbatim recent turns)?
# 2. What gets summarized into Tier 2 (episodic store)?
# 3. What gets distilled into Tier 3 (semantic facts)?
# 4. At session N+1, how does the agent know where it left off?

def retrieve_context(
    query: str,
    user_id: str,
    max_tokens: int,
    memory: MemoryManager
) -> str:
    """
    Retrieve relevant context for a new turn.

    TODO: implement this function.
    - Tier 1: always include recent N turns
    - Tier 2: retrieve episodic memories by cosine similarity to query
    - Tier 3: prepend high-confidence facts about the user/project
    - Stay within max_tokens budget
    """
    raise NotImplementedError("Your turn!")

# Bonus: simulate "context rot" — add 20 turns and observe quality degradation.
# Then implement a rolling summary that prevents it.